In [34]:
import torch

print("PyTorch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

print("Using:", device)

PyTorch: 2.14.1
MPS available: True
Using: mps


## 初始化与MiniMax

In [35]:
import numpy as np


def initial_state():
    board = np.zeros((3, 3), dtype=int)  # 提示：np.zeros((行数, 列数), dtype=int)
    current_player = 1
    return board, current_player


# board, player = initial_state()

# board[1, 1] = 1   # 把当前玩家的棋子放到中央
# player = -1            # 切换到另一位玩家

# for row, col in [(1, 1), (0, 0)]:
#     board[row, col] = player   # 哪个变量存着这一手的棋子值？
#     player = -player            # 怎样把当前值的正负号翻转？

# for row, col in [(1, 1), (1, 1)]:
#     if board[row, col] == 0:
#         board[row, col] = player
#         player = -player

In [36]:
def legal_actions(board):
    moves = list()
    for row in range(3):
        for col in range(3):
            if board[row, col] == 0:
                moves.append((row, col))
    return moves

In [37]:
def has_row_win(board, player):
    for row in range(3):
        if np.all(board[row, :] == player):
            return True
    return False

In [38]:
def has_col_win(board, player):
    for col in range(3):
        if np.all(board[:, col] == player):
            return True
    return False

In [39]:
# def has_diag_win(board, player):
#     for i in np.diag(board):
#         if i != player:
#             for i in np.diag(np.fliplr(board)):
#                 if i != player:
#                     return False
#     return True
def has_diag_win(board, player):
    main_win = np.all(np.diag(board) == player)
    anti_win = np.all(np.diag(np.fliplr(board)) == player)
    return main_win or anti_win

In [40]:
def has_won(board, player):
    row = has_row_win(board, player)
    col = has_col_win(board, player)
    diag = has_diag_win(board, player)
    return row or col or diag

In [41]:
def game_result(board, player):
    if has_won(board, -player) == True:
        return -player
    elif len(legal_actions(board)) == 0:
        return 0
    return None

In [42]:
def step(board, player, row, col):
    result = game_result(board, player)
    if result != None:
        return board, player
    if 0 <= row < 3 and 0 <= col < 3 and board[row, col] == 0:
        board[row, col] = player
        player = -player
    return board, player

In [43]:
import random


def random_action(board):
    moves = random.choice(legal_actions(board))
    return moves

In [44]:
def play_random_game():
    board, player = initial_state()

    while game_result(board, player) is None:
        row, col = random_action(board)
        board, player = step(board, player, row, col)

    print(board)
    return game_result(board, player)

In [45]:
# 终局评分
def terminal_value(result, player):
    if result == 0:
        return 0
    elif result == player:
        return 1
    else:
        return -1

In [46]:
# 从当前局面出发，假设双方都走得最好，轮到行动的玩家最终能拿几分？当棋盘上有好几个空格时，这个函数就更有用了。它会试走每个合法位置，再考虑对手会怎样应对。每条走法一旦走到终局，你写的 terminal_value() 就能给它打分。
def minimax_value(board, player):
    result = game_result(board, player)
    if result is not None:
        return terminal_value(result, player)
    scores = []
    for row, col in legal_actions(board):  # 是否有问题，遍历怎么能称为“走得最好”
        child_board, next_player = step(board.copy(), player, row, col)
        child_score = minimax_value(child_board, next_player)
        scores.append(-child_score)
    return max(scores)

In [47]:
def minimax_action(board, player):
    moves = legal_actions(board)
    scores = []
    for row, col in moves:
        child_board, next_player = step(board.copy(), player, row, col)
        child_score = minimax_value(child_board, next_player)
        scores.append(-child_score)

    best_index = scores.index(max(scores))
    return moves[best_index]

In [48]:
def play_minimax_vs_random():
    board, player = initial_state()

    while game_result(board, player) is None:
        if player == -1:
            row, col = minimax_action(board, player)  # O 用 Minimax
        else:
            row, col = random_action(board)  # X 随机选
        board, player = step(board, player, row, col)

    print(board)
    return game_result(board, player)

In [49]:
def play_minimax_vs_minimax():
    board, player = initial_state()

    while game_result(board, player) is None:
        row, col = minimax_action(board, player)
        board, player = step(board, player, row, col)

    print(board)
    return game_result(board, player)

## MTCS

S4 整段要解决的是一个问题：轮到我落子，怎样用有限次试走挑出一步棋？ 先学一次试走怎样得到分数，再学怎样给不同走法分别记分、决定下一次试哪步。等这件事弄清楚，再把对手的应对也存进树里，最后让搜索和随机棋手、Minimax 实际下棋。

In [50]:
test_board = np.array(
    [
        [-1, -1, 0],  # O O ·
        [-1, 1, 1],  # O X X
        [0, 1, 1],  # · X X
    ]
)

In [51]:
# 复制棋盘，让双方随机下到终局，返回起始玩家视角的得分，赢 +1、平 0、输 -1。
def rollout(board, player):
    board = board.copy()
    start_player = player

    while game_result(board, player) is None:
        row, col = random_action(board)
        board, player = step(board, player, row, col)

    result = game_result(board, player)
    return terminal_value(result, start_player)


print(rollout(test_board, -1))
print(legal_actions(test_board))

1
[(0, 2), (2, 0)]


In [52]:
practice_board = np.array(
    [
        [-1, -1, 0],  # O O ·
        [1, 0, 1],  # X · X
        [-1, 1, 1],  # O X X
    ]
)

In [53]:
# 从同一盘棋出发，随机续下 20 次，看看 O 平均每局得几分。
practice_scores = []
for _ in range(20):
    practice_scores.append(rollout(practice_board, -1))

practice_n = len(practice_scores)
practice_w = sum(practice_scores)  # 这批模拟中 O 的累计得分，可以用 sum
practice_q = practice_w / practice_n  # O 平均每局得几分

print(practice_scores)
print(practice_n, practice_w, practice_q)

[1, 1, 1, -1, 1, 1, 1, -1, -1, -1, -1, 1, -1, -1, -1, 1, 1, 1, 1, 1]
20 4 0.2


In [54]:
# 上一步的Q把右上和中心两种走法合在了一起。现在给每个候选落点分别打分
for row, col in legal_actions(practice_board):
    practice_child, practice_next = step(practice_board.copy(), -1, row, col)
    move_scores = []

    for _ in range(5):
        child_score = rollout(practice_child, practice_next)
        move_scores.append(-child_score)  # 用 child_score 算出 O 选择这一步的得分

    move_n = len(move_scores)
    move_w = sum(move_scores)
    print((row, col), move_n, move_w, move_w / move_n)

(0, 2) 5 5 1.0
(1, 1) 5 -5 -1.0


In [55]:
# 先走指定动作，再调用 rollout。得分为输入player的得分。
def simulate_move(board, player, move, stats):
    # 在副本上走出指定的第一手
    row, col = move
    child_board, next_player = step(board.copy(), player, row, col)

    # 从子局面模拟到底，把得分换回选择 move 的玩家视角
    child_score = rollout(child_board, next_player)
    move_score = -child_score

    # 找到这个落点原来的记录，接着累计
    record = stats[move]
    record["N"] += 1
    record["W"] += move_score

In [56]:
practice_stats = {
    (0, 2): {"N": 5, "W": 5},
    (1, 1): {"N": 5, "W": -5},
}

# 算出每个落点对应的平均得分
practice_q_by_move = {}
for move, record in practice_stats.items():
    practice_q_by_move[move] = record["W"] / record["N"]

# 比较两个落点的 Q，取出 Q 较大的那个坐标
practice_next_move = max(
    practice_q_by_move, key=practice_q_by_move.get
)  # max(..., key=...) 是 Python 内置函数 max() 的高级用法：1遍历practice_q_by_move的键；2key=practice_q_by_move.get：映射key对应的值；3比较；4返回最大值的键

simulate_move(practice_board, -1, practice_next_move, practice_stats)

print(practice_next_move)
print(practice_stats)
print(practice_q_by_move)

(0, 2)
{(0, 2): {'N': 6, 'W': 6}, (1, 1): {'N': 5, 'W': -5}}
{(0, 2): 1.0, (1, 1): -1.0}


In [57]:
import numpy as np


# select_uct 解决的是“下一次模拟花在哪一步”。优先选择未模拟过的动作，其余情况根据 Q + bonus 安排下一次模拟。
def select_uct(board, stats):
    moves = legal_actions(board)

    # 优先选择未模拟过的动作
    for move in moves:
        if stats[move]["N"] == 0:
            return move

    # 所有动作都已模拟过，比较 UCT 评分。UCT 分数 = 平均得分 Q + 探索加分 √(ln(total_N) / N)
    total_n = sum(stats[move]["N"] for move in moves)
    priorities = {}

    for move in moves:
        record = stats[move]
        n = record["N"]
        q = record["W"] / n
        bonus = np.sqrt(np.log(total_n) / n)
        priorities[move] = q + bonus

    return max(priorities, key=priorities.get)

In [58]:
print("第 12 次试", select_uct(practice_board, practice_stats))

both_win_stats = {
    (0, 2): {"N": 6, "W": 6},
    (2, 0): {"N": 1, "W": 1},
}
print("两步都赢时试", select_uct(test_board, both_win_stats))

第 12 次试 (0, 2)
两步都赢时试 (2, 0)


In [59]:
trial_stats = {}
for move in legal_actions(practice_board):
    trial_stats[move] = {"N": 0, "W": 0}

for trial in range(1, 4):
    trial_move = select_uct(practice_board, trial_stats)
    simulate_move(practice_board, -1, trial_move, trial_stats)
    print("第", trial, "次试走", trial_move, trial_stats[trial_move])

recommended_move = max(trial_stats, key=lambda move: trial_stats[move]["N"])
print("最终记录", trial_stats)
print("推荐落子", recommended_move)

第 1 次试走 (0, 2) {'N': 1, 'W': 1}
第 2 次试走 (1, 1) {'N': 1, 'W': -1}
第 3 次试走 (0, 2) {'N': 2, 'W': 2}
最终记录 {(0, 2): {'N': 2, 'W': 2}, (1, 1): {'N': 1, 'W': -1}}
推荐落子 (0, 2)


In [60]:
# 先定义 make_node。一个节点保存一张棋盘和当前行动方。`stats` 记“在这里走某一步，试过几次、得了几分”，`children` 记“走出这一步，会到哪个节点”。
def make_node(board, player):
    stats = {}
    if game_result(board, player) is None:
        for move in legal_actions(board):
            stats[move] = {"N": 0, "W": 0}

    return {
        "board": board.copy(),
        "player": player,
        "stats": stats,
        "children": {},
    }

In [61]:
# 为什么还要把对手的应对存进树里：要让 X 在搜索时考虑到“O 也会挑好棋”
# 轮到 X             X 走左中后，轮到 O
# O O ·              O O ·
# · · X       →      X · X
# O X X              O X X
before_x = practice_board.copy()
before_x[1, 0] = 0
practice_root = make_node(before_x, 1)

x_move = (1, 0)
row, col = x_move
child_board, child_player = step(
    practice_root["board"].copy(), practice_root["player"], row, col
)

practice_root["children"][x_move] = make_node(child_board, child_player)

o_node = practice_root["children"][x_move]
print(np.array_equal(o_node["board"], practice_board))
print(o_node["player"], o_node["stats"])

True
-1 {(0, 2): {'N': 0, 'W': 0}, (1, 1): {'N': 0, 'W': 0}}


In [62]:
# 搜索每走一步，要先看这个落点有没有对应的子节点。第一次走这条分支，就创建并保存子节点。再次从同一局面走这一步，就取回先前保存的节点。 
def expand(node, move):
    if move in node["children"]:
        return node["children"][move]

    row, col = move
    child_board, child_player = step(
        node["board"].copy(), node["player"], row, col
    )
    child = make_node(child_board, child_player)
    node["children"][move] = child
    return child

In [ ]:
# 测试expand
saved_o = expand(practice_root, (1, 0))
o_move = select_uct(saved_o["board"], saved_o["stats"])
practice_leaf = expand(saved_o, o_move)

print(saved_o is o_node)
print(o_move, game_result(practice_leaf["board"], practice_leaf["player"]))
print(expand(saved_o, o_move) is practice_leaf)

True
(0, 2) -1
True


In [ ]:

def backpropagate(path, leaf_score):
    score = leaf_score

    for node, move in reversed(path):# 倒着遍历path
        score = -score  # 把手里的分数换成选择 move 的玩家的视角
        record = node["stats"][move]
        record["N"] += 1
        record["W"] += score


practice_path = [
    (practice_root, (1, 0)),  # X 走左中
    (saved_o, o_move),        # O 回右上
]

practice_leaf_score = rollout(practice_leaf["board"], practice_leaf["player"])
backpropagate(practice_path, practice_leaf_score)

print("O 走右上", saved_o["stats"][o_move])
print("X 走左中", practice_root["stats"][(1, 0)])

O 走右上 {'N': 1, 'W': 1}
X 走左中 {'N': 1, 'W': -1}


In [57]:
def search_action(board, player, num_simulations):
    if game_result(board, player) is not None:
        return None

    root = make_node(board, player)

    for i in range(num_simulations):
        mcts_simulation(root)
        # 补一行，以 root 为起点运行一次 mcts_simulation

    moves = legal_actions(board)
    visits = []

    for move in moves:
        record = root["stats"][move]
        visits.append(record["N"])

    best_index = visits.index(max(visits))
    return moves[best_index]

In [59]:
import random

defense_board = np.array(
    [
        [1, 1, 0],  # X X .
        [0, -1, 0],  # . O .
        [0, 0, 0],  # . . .
    ]
)
before = defense_board.copy()

print("Minimax", minimax_action(defense_board, -1))

random.seed(0)
print("MCTS", search_action(defense_board, -1, 500))
print(np.array_equal(defense_board, before))

Minimax (0, 2)
MCTS (0, 2)
True


In [60]:
reference_move = minimax_action(defense_board, -1)

for num_simulations in [10, 50, 100, 500]:
    correct_count = 0

    for seed in range(10):
        random.seed(seed)

        move = search_action(defense_board, -1, num_simulations)
        # 调用 search_action，得到 move

        if move == reference_move:
            correct_count = correct_count + 1

    print(num_simulations, correct_count, "/10")

10 8 /10
50 10 /10
100 10 /10
500 10 /10


In [61]:
def play_mcts_vs_random(num_simulations):
    board, player = initial_state()

    while game_result(board, player) is None:
        if player == 1:
            row, col = search_action(board, player, num_simulations)
            # 调用 search_action，得到 row、col
        else:
            row, col = random_action(board)

        board, player = step(board, player, row, col)

    return board, game_result(board, player)

In [62]:
random.seed(0)

final_board, result = play_mcts_vs_random(100)
print(final_board)
print(result)

[[-1  0  1]
 [ 0 -1 -1]
 [ 1  1  1]]
1


In [63]:
counts = {1: 0, 0: 0, -1: 0}

for seed in range(10):
    random.seed(seed)
    final_board, result = play_mcts_vs_random(100)

    counts[result] = counts[result] + 1
    # 补一行，累计本局赛果

print(counts)

{1: 10, 0: 0, -1: 0}


In [64]:
def play_mcts_vs_random(num_simulations, mcts_player=1):
    board, player = initial_state()

    while game_result(board, player) is None:
        if player == mcts_player:  # 修改这一行，使用 mcts_player 判断
            row, col = search_action(board, player, num_simulations)
        else:
            row, col = random_action(board)

        board, player = step(board, player, row, col)

    return board, game_result(board, player)

In [65]:
random.seed(0)

final_board, result = play_mcts_vs_random(100, mcts_player=-1)
print(final_board)
print(result)

[[-1  0  1]
 [ 1 -1  0]
 [ 1  0 -1]]
-1


In [66]:
for mcts_player in [1, -1]:
    # 每换一次执棋方，重新统计
    counts = {1: 0, 0: 0, -1: 0}

    for seed in range(10):
        random.seed(seed)

        # 完整下一局，得到原始赛果
        final_board, result = play_mcts_vs_random(100, mcts_player)

        # 换成 MCTS 的视角：赢 1，平 0，输 -1
        score = terminal_value(result, mcts_player)

        counts[score] = counts[score] + 1

    print("MCTS执棋方", mcts_player, counts)

MCTS执棋方 1 {1: 10, 0: 0, -1: 0}
MCTS执棋方 -1 {1: 10, 0: 0, -1: 0}


In [67]:
def play_mcts_vs_minimax(num_simulations, mcts_player=1):
    board, player = initial_state()

    while game_result(board, player) is None:
        if player == mcts_player:
            row, col = search_action(board, player, num_simulations)
        else:
            row, col = minimax_action(board, player)

        board, player = step(board, player, row, col)

    return board, game_result(board, player)

In [68]:
random.seed(0)

final_board, result = play_mcts_vs_minimax(100, mcts_player=1)

print(final_board)
print(result)

[[-1 -1  1]
 [ 1  1 -1]
 [-1  1  1]]
0


In [69]:
# for mcts_player in [1, -1]:
#     counts = {1: 0, 0: 0, -1: 0}

#     for seed in range(10):
#         random.seed(seed)

#         final_board, result = play_mcts_vs_minimax(500, mcts_player)
#         score = terminal_value(result, mcts_player)

#         counts[score] = counts[score] + 1

#     print("MCTS执棋方", mcts_player, counts)

## 双头网络

In [70]:
import numpy as np


# 将棋盘按照player拆成两个，输出一个+1维的numpy数组
def encode_board(board, player):
    own_plane = (board == player).astype(np.float32)
    opponent_plane = (board == -player).astype(np.float32)
    return np.stack([own_plane, opponent_plane], axis=0)


defense_board = np.array(
    [
        [1, 1, 0],
        [0, -1, 0],
        [0, 0, 0],
    ]
)

encoded = encode_board(defense_board, player=-1)
print(encoded.shape, encoded.dtype, type(encoded))
# print(encoded[0])
# print(encoded[1])
# print(encoded)

(2, 3, 3) float32 <class 'numpy.ndarray'>


In [71]:
import torch


# 讲numpy数组变成tensor，且再+1维。unsqueeze只能加1维，即套一层壳。player是谁就把谁的视角的board放在前面
def board_to_input(board, player):
    encoded = encode_board(board, player)  # NumPy 数组，形状 (2, 3, 3)
    x = torch.from_numpy(encoded)  # 转成 PyTorch 张量，形状和数值都保留
    return x.unsqueeze(0)  # 在指定位置插入一个长度为 1 的维度


x = board_to_input(defense_board, player=-1)
# print(x.shape, x.dtype)
# print(x[0, :, 0, 0].tolist())
print(x)

tensor([[[[0., 0., 0.],
          [0., 1., 0.],
          [0., 0., 0.]],

         [[1., 1., 0.],
          [0., 0., 0.],
          [0., 0., 0.]]]])


In [72]:
from torch import nn

# 把每个局面的 2 × 3 × 3 个数展平成 18 个数，再用一层 nn.Linear(18, 32) 把它们组合成 32 个特征。这里的 32 是我们给小网络选的宽度。随后，两个头各自读取这组特征，回答自己的问题。让输入沿这些层算出输出，就叫一次前向传播。
shared_layer = nn.Linear(18, 32)
policy_head = nn.Linear(
    32, 9
)  # 策略头给每个格子一个分数，供搜索判断哪些走法值得优先探索。
value_head = nn.Linear(
    32, 1
)  # 价值头给整个局面一个分数，估计从当前行动方 O 的角度，走到终局能拿多少分。为什么是O，因为例子用的defence_board马上要O走了。

flat = x.flatten(start_dim=1)  # 保留第 0 维的局面数，（1，2，3，3），从“2”开始展开
features = torch.relu(shared_layer(flat))  # ReLU 把负数置零，保留正数(激活函数)
logits = policy_head(features)
value = torch.tanh(value_head(features))  # 把估值限制在 [-1, 1](激活函数)

print(logits.shape, value.shape)
print("右上角的落子分数", logits[0, 2].item())
print("从 O 看整盘棋的估值", value[0, 0].item())
print(logits)
print(value_head(features))

torch.Size([1, 9]) torch.Size([1, 1])
右上角的落子分数 -0.19594073295593262
从 O 看整盘棋的估值 0.14705100655555725
tensor([[ 0.0797,  0.2324, -0.1959,  0.0115,  0.1564,  0.0565,  0.2100,  0.0531,
         -0.1468]], grad_fn=<AddmmBackward0>)
tensor([[0.1481]], grad_fn=<AddmmBackward0>)


logits相较于value少做了一次激活，这是故意设计的吗
后续要做softmax，激活后会破坏；value_head要把估值限制在 [-1, 1]。

In [73]:
# 把logits这九个分数变成九个概率：已占格的概率为 0，所有空格的概率加起来是 1。
def legal_policy(board, move_logits):
    cells = torch.as_tensor(
        board, device=move_logits.device
    ).flatten()  # torch.as_tensor(...)：PyTorch 工厂函数，将 board（可能是 list、numpy array 或已有 tensor）转换为 PyTorch Tensor。
    masked_logits = move_logits.masked_fill(
        cells != 0, float("-inf")
    )  # .masked_fill(condition, value)：PyTorch 方法，在 condition 为 True 的位置将值替换为 value，其余位置保持不变
    return torch.softmax(masked_logits, dim=0)


test_logits = torch.zeros_like(
    logits[0]
)  # 一个局面的九个分数。torch.zeros_like(tensor)：PyTorch 工厂函数，创建一个与输入张量形状、dtype、device 完全相同的全零张量
test_logits[0] = 10.0  # 左上角，已有 X（测试用）
test_logits[2] = 1.0  # 右上角，空格（测试用）

priors = legal_policy(defense_board, test_logits)
print(priors)
print(priors.reshape(3, 3))
print(priors.sum().item())

tensor([0.0000, 0.0000, 0.3522, 0.1296, 0.0000, 0.1296, 0.1296, 0.1296, 0.1296])
tensor([[0.0000, 0.0000, 0.3522],
        [0.1296, 0.0000, 0.1296],
        [0.1296, 0.1296, 0.1296]])
1.0


In [74]:
# 前向计算
def forward_batch(x_batch):
    flat = x_batch.flatten(start_dim=1)
    features = torch.relu(shared_layer(flat))
    logits = policy_head(features)
    values = torch.tanh(value_head(features))
    return logits, values


# step 会原地落子，传入副本才能同时保留落子前后的棋盘
after_board, after_player = step(defense_board.copy(), -1, 0, 2)

before_x = board_to_input(defense_board, -1)
after_x = board_to_input(after_board, after_player)
x_batch = torch.cat(
    [before_x, after_x], dim=0
)  # PyTorch 提供的张量拼接函数，用于将多个张量沿某一维度连接起来。

batch_logits, batch_values = forward_batch(
    x_batch
)  # 为什么多个batch可以进入shared_layer：nn.Linear(in_features, out_features) 接受形状为 (*, H_in) 的输入，其中 * 代表零个或多个任意大小的批量维度，H_in 必须严格等于 in_features。输出形状为 (*, H_out)。(2, 18) 的含义是表示一个包含 2 个样本的批次（batch），每个样本的特征维度为 18。这两个样本会在同一次前向传播中并行计算，共享同一组权重矩阵。
before_p = legal_policy(defense_board, batch_logits[0])
after_p = legal_policy(after_board, batch_logits[1])

print(x_batch.shape, batch_logits.shape, batch_values.shape)
print("右上角落子前后的概率", before_p[2].item(), after_p[2].item())
print(batch_values)

torch.Size([2, 2, 3, 3]) torch.Size([2, 9]) torch.Size([2, 1])
右上角落子前后的概率 0.13602697849273682 0.0
tensor([[0.1471],
        [0.1821]], grad_fn=<TanhBackward0>)


In [75]:
sample_board, turn = step(after_board.copy(), after_player, 2, 2)
sample_board, turn = step(sample_board, turn, 2, 0)
result = game_result(sample_board, turn)

players = [-1, after_player]  # 顺序与 x_batch 的两行一致
scores = [terminal_value(result, player) for player in players]
targets = batch_values.new_tensor(scores).reshape(
    -1, 1
)  # new_tensor(data) 是 PyTorch Tensor 对象的一个实例方法。以 batch_values 为模板，创建一个新张量。新张量会自动继承 batch_values 的以下属性：dtype（数据类型，如 float32）、device（设备，如 cuda:0 / cpu）、layout。reshape(shape) 也是 Tensor 的方法，返回一个指定形状的新视图（view）。-1 的含义：表示该维度的大小由 PyTorch 自动推断。

value_loss = (batch_values - targets).square().mean()

print("终局赛果", result)
print("两行的预测", batch_values.tolist())  # .tolist()任意形状的张量
print("两行的目标", targets.tolist())
print("价值损失", value_loss.item())  # .item()仅含1个元素的张量

终局赛果 -1
两行的预测 [[0.14705099165439606], [0.18213826417922974]]
两行的目标 [[1.0], [-1.0]]
价值损失 1.0624864101409912


In [76]:
shared_layer.zero_grad()  # 清掉累积的参数梯度，方便重跑这一格
value_head.zero_grad()

_, values = forward_batch(x_batch)
values.retain_grad()  # 把中间结果 values 的梯度也留下来，方便观察
value_loss = (values - targets).square().mean()

value_loss.backward()  # backward() 会从损失出发，沿着刚才的前向计算往回求导，一直算到价值头和共享层的每个参数。优化器下一步就能用这些梯度调整权重。

print("损失对 O、X 两个估值的梯度", values.grad.flatten().tolist())
# values.grad获取 values 张量的梯度（一个与 values 形状相同的 torch.Tensor）
print("价值头权重的梯度形状", value_head.weight.grad.shape)
# .weight：访问该模块的权重参数

损失对 O、X 两个估值的梯度 [-0.8529490232467651, 1.182138204574585]
价值头权重的梯度形状 torch.Size([1, 32])


In [77]:
# 优化器怎么用不知道，背后什么原理也不知道
params = list(shared_layer.parameters()) + list(value_head.parameters())
optimizer = torch.optim.SGD(params, lr=0.01)

optimizer.zero_grad()  # 清掉上一格的梯度，这次用当前参数重新计算
_, values_before = forward_batch(x_batch)
loss_before = (values_before - targets).square().mean()
loss_before.backward()
optimizer.step()  # 用刚算出的参数梯度更新一次

with torch.no_grad():
    _, values_after = forward_batch(x_batch)
    loss_after = (values_after - targets).square().mean()

print("更新前", values_before.flatten().tolist(), loss_before.item())
print("更新后", values_after.flatten().tolist(), loss_after.item())

更新前 [0.14705099165439606, 0.18213826417922974] 1.0624864101409912
更新后 [0.14890044927597046, 0.15897217392921448] 1.0337934494018555


In [78]:
# 面对这个局面，搜索更倾向于走哪一步
# 把每个格子分到了多少比例的访问也保留下来。
def policy_target_from_root(root):
    visits = torch.zeros((3, 3), dtype=torch.float32)  # 为什么要dtype=
    for move, record in root["stats"].items():
        visits[move] = record["N"]

    total = visits.sum()
    if total.item() == 0:
        raise ValueError("先运行至少一次模拟，再计算策略目标")
    return (visits / total).flatten()  # 展平成与策略头九个输出对应的顺序


two_empty = np.array(
    [
        [1, 0, 1],
        [1, 1, -1],
        [-1, -1, 0],
    ]
)
root = make_node(two_empty, -1)
for _ in range(3):
    mcts_simulation(root)

pi = policy_target_from_root(root)
print(pi)
print(pi.reshape(3, 3))
print(pi.sum().item())

tensor([0.0000, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.6667])
tensor([[0.0000, 0.3333, 0.0000],
        [0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.6667]])
1.0


In [ ]:
# 计算策略损失。例：策略损失 = (-ln(p1) - ln(p8) - ln(p8)) / 3= -(1/3 × ln(p1) + 2/3 × ln(p8))
def policy_loss_for_board(board, move_logits, target_pi):
    cells = torch.as_tensor(board, device=move_logits.device).flatten()
    legal = cells == 0
    log_p = torch.log_softmax(move_logits[legal], dim=0)
    pi_legal = target_pi.to(move_logits)[legal]
    return -(log_p * pi_legal).sum()


two_empty_logits, _ = forward_batch(board_to_input(two_empty, -1))
move_logits = two_empty_logits[0]
p = legal_policy(two_empty, move_logits)

equal_logits = torch.zeros_like(move_logits)
print(
    "两个合法动作等分时的损失",
    policy_loss_for_board(two_empty, equal_logits, pi).item(),
)
print("网络给上中、右下的概率", p[[1, 8]].tolist())
print("网络的策略损失", policy_loss_for_board(two_empty, move_logits, pi).item())

两个合法动作等分时的损失 0.6931471824645996
网络给上中、右下的概率 [0.47054368257522583, 0.5294562578201294]
网络的策略损失 0.6752252578735352


In [ ]:
policy_params = list(shared_layer.parameters()) + list(policy_head.parameters())
policy_optimizer = torch.optim.SGD(policy_params, lr=0.01)

x_o = board_to_input(two_empty, -1)
policy_optimizer.zero_grad()
batch_logits, _ = forward_batch(x_o)
scores = batch_logits[0]
scores.retain_grad()

p8_before = legal_policy(two_empty, scores)[8].item()
loss_before = policy_loss_for_board(two_empty, scores, pi)
loss_before.backward()
print("上中、右下的分数梯度", scores.grad[[1, 8]].tolist())

policy_optimizer.step()

with torch.no_grad():
    new_logits, _ = forward_batch(x_o)
    p8_after = legal_policy(two_empty, new_logits[0])[8].item()
    loss_after = policy_loss_for_board(two_empty, new_logits[0], pi)

print("右下角概率，更新前后", p8_before, p8_after)
print("策略损失，更新前后", loss_before.item(), loss_after.item())

上中、右下的分数梯度 [0.1372103989124298, -0.1372104287147522]
右下角概率，更新前后 0.5294562578201294 0.5337083339691162
策略损失，更新前后 0.6752252578735352 0.6729184985160828


In [ ]:
sample_player = -1
end_board, next_player = step(two_empty.copy(), sample_player, 2, 2)
result = game_result(end_board, next_player)
z = terminal_value(result, sample_player)


def losses_for_sample(board, player, target_pi, target_value):
    logits, values = forward_batch(board_to_input(board, player))
    policy_loss = policy_loss_for_board(board, logits[0], target_pi)
    value_loss = (values[0, 0] - target_value).square()
    total_loss = policy_loss + value_loss
    return policy_loss, value_loss, total_loss


# policy_params 已经包含共享层和策略头，加上价值头就集齐了三层
joint_optimizer = torch.optim.SGD(
    policy_params + list(value_head.parameters()), lr=0.01
)

joint_optimizer.zero_grad()
p_before, v_before, total_before = losses_for_sample(two_empty, sample_player, pi, z)
total_before.backward()
joint_optimizer.step()

with torch.no_grad():
    p_after, v_after, total_after = losses_for_sample(two_empty, sample_player, pi, z)

print("O 的价值目标", z)
print(
    "更新前（策略、价值、总损失）",
    p_before.item(),
    v_before.item(),
    total_before.item(),
)
print(
    "更新后（策略、价值、总损失）", p_after.item(), v_after.item(), total_after.item()
)

O 的价值目标 1
更新前（策略、价值、总损失） 0.6729184985160828 0.5891447067260742 1.2620632648468018
更新后（策略、价值、总损失） 0.6707260608673096 0.4611002802848816 1.131826400756836


In [ ]:
@torch.no_grad()  # torch.no_grad() 是 PyTorch 提供的上下文管理器/装饰器。它的作用是在该函数执行期间禁用梯度计算。
def evaluate_for_search(board, player):
    result = game_result(board, player)
    if result is not None:
        return {}, float(terminal_value(result, player))

    logits, values = forward_batch(board_to_input(board, player))
    p_grid = legal_policy(board, logits[0]).reshape(3, 3)

    priors = {}
    for move in legal_actions(board):
        priors[move] = p_grid[
            move
        ].item()  # 从 p_grid 取出这个落点的概率，转成 Python 浮点数

    return priors, values[0, 0].item()


priors, v = evaluate_for_search(two_empty, sample_player)
print("落子前的先验", priors, "概率和", sum(priors.values()))
print("落子前 O 的估值", v)
print("落子后的终局", evaluate_for_search(end_board, next_player))

落子前的先验 {(0, 1): 0.46212494373321533, (2, 2): 0.5378749966621399} 概率和 0.9999999403953552
落子前 O 的估值 0.3209563195705414
落子后的终局 ({}, -1.0)
